# Cours : Résolution des Équations du 3ème et 4ème Degré

Ce notebook rassemble la théorie mathématique et les exercices d'implémentation en Python pour la résolution analytique des polynômes de degré 3 (méthode de Cardan) et de degré 4 (méthode de Ferrari).

---

## 1. Équations du 3ème degré (Méthode de Cardan)

Soit l'équation générale du troisième degré :
$$a x^3 + b x^2 + c x + d = 0 \quad (a \neq 0)$$

### Étape 1 : Réduction de l'équation (Transformation de Tschirnhaus)
On pose $x = t - \frac{b}{3a}$ pour éliminer le terme de degré 2. L'équation devient dépressionnaire :
$$t^3 + p t + q = 0$$
avec :
$$p = \frac{3ac - b^2}{3a^2} \quad \text{et} \quad q = \frac{2b^3 - 9abc + 27a^2d}{27a^3}$$

### Étape 2 : Substitution de Cardan
On pose $t = u + v$ sous la contrainte $uv = -\frac{p}{3}$. L'équation donne le système :
$$\begin{cases} u^3 + v^3 = -q \\ u^3 v^3 = -\frac{p^3}{27} \end{cases}$$

### Étape 3 : Discriminant et résolution
$U = u^3$ et $V = v^3$ sont les racines de $Z^2 + qZ - \frac{p^3}{27} = 0$. Le discriminant réduit est :
$$\Delta = \left(\frac{q}{2}\right)^2 + \left(\frac{p}{3}\right)^3$$

- **$\Delta > 0$** : 1 racine réelle $t_1 = u + v$.
- **$\Delta = 0$** : Racines réelles multiples ($t_1 = 2u, t_2 = t_3 = -u$).
- **$\Delta < 0$** : 3 racines réelles distinctes via la formule trigonométrique :
  $$t_k = 2\sqrt{-\frac{p}{3}} \cos\left(\frac{\theta + 2k\pi}{3}\right) \quad \text{avec} \quad \cos(3\theta) = \frac{3q}{2p \sqrt{-\frac{p}{3}}}$$

In [ ]:
from math import cos, pi


def solve_cubic(a, b, c, d):
    """
    Résolution d'une équation a*x^3 + b*x^2 + c*x + d = 0 par la méthode de Cardan.
    """

    
    # 1. Forme réduite t^3 + p*t + q = 0
    p = ...
    q = ...
    shift = b / (3 * a)

    # 2. Discriminant
    delta = ...

    # Compléter la résolution selon le signe de delta
    # - Si delta > 1e-11
    # - Si abs(delta) <= 1e-11
    # - Si delta < -1e-11
    
    pass

# --- Tests ---
# print(solve_cubic(1, -6, 11, -6))  # Attendu: [1, 2, 3]
# print(solve_cubic(1, -3, 3, -1))   # Attendu: [1, 1, 1]
# print(solve_cubic(1, 0, 3, -4))    # Attendu: [1, -0.5+1.93i, -0.5-1.93i]

In [ ]:
import cmath
import math

def solve_cubic_solution(a: float, b: float, c: float, d: float) -> list:
    if a == 0:
        raise ValueError("Le coefficient 'a' doit être non nul.")
    
    p = (3 * a * c - b**2) / (3 * a**2)
    q = (2 * b**3 - 9 * a * b * c + 27 * a**2 * d) / (27 * a**3)
    shift = b / (3 * a)

    delta = (q / 2)**2 + (p / 3)**3
    eps = 1e-11

    def cbrt(x):
        return math.copysign(abs(x)**(1/3), x)

    if delta > eps:
        u = cbrt(-q/2 + math.sqrt(delta))
        v = cbrt(-q/2 - math.sqrt(delta))
        t1 = u + v
        t2 = complex(-(u + v) / 2, (u - v) * math.sqrt(3) / 2)
        t3 = complex(-(u + v) / 2, -(u - v) * math.sqrt(3) / 2)
        roots = [t1, t2, t3]
    elif abs(delta) <= eps:
        if abs(p) < eps and abs(q) < eps:
            roots = [0.0, 0.0, 0.0]
        else:
            u = cbrt(-q/2)
            roots = [2 * u, -u, -u]
    else:
        r = math.sqrt(-p / 3)
        cos_3theta = max(-1.0, min(1.0, 3 * q / (2 * p * r)))
        theta = math.acos(cos_3theta) / 3
        roots = [
            2 * r * math.cos(theta),
            2 * r * math.cos(theta + 2 * math.pi / 3),
            2 * r * math.cos(theta + 4 * math.pi / 3)
        ]

    return sorted([r - shift for r in roots], key=lambda z: (z.real if isinstance(z, complex) else z, z.imag if isinstance(z, complex) else 0))

print("Degré 3 (1, -6, 11, -6) ->", solve_cubic_solution(1, -6, 11, -6))

## 2. Équations du 4ème degré (Méthode de Ferrari)

Soit l'équation générale du quatrième degré :
$$a x^4 + b x^3 + c x^2 + d x + e = 0 \quad (a \neq 0)$$

### Étape 1 : Réduction
On pose $x = t - \frac{b}{4a}$. L'équation devient :
$$t^4 + p t^2 + q t + r = 0$$
avec :
$$p = \frac{8ac - 3b^2}{8a^2}, \quad q = \frac{b^3 - 4abc + 8a^2d}{8a^3}, \quad r = \frac{-3b^4 + 16ab^2c - 64a^2bd + 256a^3e}{256a^4}$$

### Étape 2 : La Résolvante Cubique
On introduit un paramètre $y$ pour compléter le carré. L'annulation du discriminant de la forme quadratique conduit à la **résolvante cubique** :
$$8 y^3 + 8 p y^2 + (2 p^2 - 8 r) y - q^2 = 0$$

### Étape 3 : Factorisation
1. Résoudre la résolvante pour obtenir une racine $y_0 > 0$.
2. Décomposer en un produit de deux équations du second degré en $t$ :
   $$t^2 - \sqrt{2y_0}\,t + \left(\frac{p}{2} + y_0 + \frac{q}{2\sqrt{2y_0}}\right) = 0$$
   $$t^2 + \sqrt{2y_0}\,t + \left(\frac{p}{2} + y_0 - \frac{q}{2\sqrt{2y_0}}\right) = 0$$
3. Obtenir les 4 racines $t_k$ et revenir aux racines de départ $x_k = t_k - \frac{b}{4a}$.

In [ ]:
def solve_quartic(a: float, b: float, c: float, d: float, e: float) -> list:
    """
    Résolution d'une équation a*x^4 + b*x^3 + c*x^2 + d*x + e = 0 par la méthode de Ferrari.
    """
    if a == 0:
        raise ValueError("Le coefficient 'a' doit être non nul.")

    p = (8*a*c - 3*b**2) / (8*a**2)
    q = (b**3 - 4*a*b*c + 8*a**2*d) / (8*a**3)
    r = (-3*b**4 + 16*a*b**2*c - 64*a**2*b*d + 256*a**3*e) / (256*a**4)
    shift = b / (4 * a)

    # TODO:
    # 1. Traiter le cas q == 0 (équation bicarrée)
    # 2. Réaliser la résolvante cubique 8*y^3 + 8*p*y^2 + (2*p^2 - 8*r)*y - q^2 = 0
    # 3. Résoudre et extraire les 4 racines
    
    pass

# --- Tests ---
# print(solve_quartic(1, -10, 35, -50, 24)) # Attendu: [1, 2, 3, 4]

In [ ]:
def solve_quartic_solution(a: float, b: float, c: float, d: float, e: float) -> list:
    if a == 0:
        raise ValueError("Le coefficient 'a' doit être non nul.")

    p = (8*a*c - 3*b**2) / (8*a**2)
    q = (b**3 - 4*a*b*c + 8*a**2*d) / (8*a**3)
    r = (-3*b**4 + 16*a*b**2*c - 64*a**2*b*d + 256*a**3*e) / (256*a**4)
    shift = b / (4 * a)
    eps = 1e-11

    def solve_quad(A, B, C):
        disc = B**2 - 4*A*C
        if isinstance(disc, complex) or disc < -eps:
            sq = cmath.sqrt(disc)
        else:
            sq = math.sqrt(max(0.0, disc))
        return [(-B - sq) / (2*A), (-B + sq) / (2*A)]

    if abs(q) < eps:
        z_roots = solve_quad(1, p, r)
        t_roots = []
        for z in z_roots:
            sq_z = cmath.sqrt(z)
            t_roots.extend([-sq_z, sq_z])
    else:
        y_roots = solve_cubic_solution(8, 8*p, 2*p**2 - 8*r, -q**2)
        y0 = None
        for y in y_roots:
            if isinstance(y, (int, float)) or abs(y.imag) < eps:
                val = y.real if isinstance(y, complex) else y
                if val > eps:
                    y0 = val
                    break
        if y0 is None:
            y0 = y_roots[0].real if isinstance(y_roots[0], complex) else y_roots[0]

        alpha = cmath.sqrt(2 * y0)
        beta = p / 2 + y0
        gamma = q / (2 * alpha)
        t_roots = solve_quad(1, -alpha, beta + gamma) + solve_quad(1, alpha, beta - gamma)

    final_roots = []
    for root in t_roots:
        val = root - shift
        if abs(val.imag) < eps:
            final_roots.append(round(val.real, 8))
        else:
            final_roots.append(complex(round(val.real, 8), round(val.imag, 8)))
    return sorted(final_roots, key=lambda z: (z.real if isinstance(z, complex) else z, z.imag if isinstance(z, complex) else 0))

print("Degré 4 (1, -10, 35, -50, 24) ->", solve_quartic_solution(1, -10, 35, -50, 24))